# 15 — Calibrated vs analytic surrogate

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-systems-today/cubespec/blob/main/notebooks/15_calibrated_vs_analytic.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ai-systems-today/cubespec/main?urlpath=lab/tree/notebooks/15_calibrated_vs_analytic.ipynb) [![nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/ai-systems-today/cubespec/blob/main/notebooks/15_calibrated_vs_analytic.ipynb) [![Dashboard](https://img.shields.io/badge/open-dashboard-2563eb)](https://sensitive-spark.lovable.app)

**Abstract.** Quantify the gap between the closed-form analytic baseline and the production-default calibrated Poly2-ridge surrogate.

**Mirrors.** **Header** → surrogate-mode toggle ([Calibrated | Analytic]).


In [ ]:
# cubespec-bootstrap
# Install cubespec on first run (Colab/Binder safe; no-op if already importable).
try:
    import cubespec  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/ai-systems-today/cubespec.git'])
    import cubespec  # noqa: F401


## 1. Two surrogates, one CSP

* **Analytic**   — closed-form `σ = F/A`, `ε ≈ k·ρ^1.5`, etc. Pedagogical baseline.
* **Calibrated** — Poly2-ridge fit to a 500-row design. Production default.

Same inputs, different outputs. We quantify the gap on all three outputs.


In [ ]:
from cubespec import (
    DEFAULT_CSP, sample_independent, compute_outputs_batch, set_mode,
    rmse, mae, r2, bias,
)
import numpy as np, pandas as pd

X = sample_independent(DEFAULT_CSP, n=10_000, seed=1337)
set_mode('calibrated'); Yc = compute_outputs_batch(X)
set_mode('analytic');   Ya = compute_outputs_batch(X)
set_mode('calibrated')  # reset to default

rows = []
for j, name in enumerate(('P7_def', 'P8_strain', 'P9_compressive_strength')):
    rows.append({'output': name,
                 'mean_calibrated': Yc[:, j].mean(),
                 'mean_analytic':   Ya[:, j].mean(),
                 'RMSE': rmse(Yc[:, j], Ya[:, j]),
                 'MAE':  mae(Yc[:, j], Ya[:, j]),
                 'R²':   r2(Yc[:, j], Ya[:, j]),
                 'bias_analytic_minus_calib': bias(Yc[:, j], Ya[:, j])})
pd.DataFrame(rows)


## 2. Side-by-side P9 distributions


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.hist(Yc[:, 2], bins=50, alpha=0.55, color='#5b8def', label='calibrated')
ax.hist(Ya[:, 2], bins=50, alpha=0.55, color='#e94f64', label='analytic')
ax.set_xlabel('P9 (MPa)'); ax.set_ylabel('count')
ax.set_title('Calibrated vs analytic — P9 distribution at seed 1337')
ax.legend(); fig.tight_layout(); plt.show()


## 3. Parity scatter (every Monte-Carlo sample)


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(Yc[:, 2], Ya[:, 2], s=4, alpha=0.3, color='#5b8def')
lo = min(Yc[:, 2].min(), Ya[:, 2].min()); hi = max(Yc[:, 2].max(), Ya[:, 2].max())
ax.plot([lo, hi], [lo, hi], 'k--', lw=0.6)
ax.set_xlabel('calibrated P9 (MPa)'); ax.set_ylabel('analytic P9 (MPa)')
ax.set_title('Parity — calibrated vs analytic')
fig.tight_layout(); plt.show()


## 4. When to use each

* **Calibrated** for any reported number — it absorbs second-order effects the analytic baseline ignores.
* **Analytic** for teaching, debugging, or quick what-ifs without loading the JSON artefact.


In [ ]:
# cubespec-reproducibility-footer
import platform, sys, numpy as np, scipy, pandas as pd, cubespec
print('Reproducibility')
print('  cubespec :', cubespec.__version__)
print('  python   :', sys.version.split()[0], 'on', platform.system())
print('  numpy    :', np.__version__)
print('  scipy    :', scipy.__version__)
print('  pandas   :', pd.__version__)
